# A Holiday Is Most of a Weekend

If public sightings follow people's days off rather than whales, a federal holiday (all fall on
weekdays, on their observed dates) should look like a weekend day. `dim_date.holiday_comparison` marks
each holiday, the same weekday one week before and after it, and the Saturday and Sunday of its week;
`compared_holiday` and `compared_holiday_date` say which holiday a day belongs to, so each holiday is
set against its own comparison days. When one of those days is a holiday itself (Christmas and New
Year's Day are a week apart), the other day stands alone.

In [ ]:
%%sql
-- WHALES_WH runs the queries.
USE WAREHOUSE whales_wh

In [ ]:
%%sql
-- The star dbt builds lives in WHALES.MARTS.
USE SCHEMA whales.marts

## Each Holiday Beside Its Own Days
One row per holiday: its public records, and the average over its same-weekday days and its weekend
days. Days with no public record count as zero.

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW holiday_compare AS
WITH bay_public_sightings AS (
    SELECT
        fct_sighting.date_key,
        fct_sighting.record_id
    FROM fct_sighting
    JOIN dim_location ON dim_location.location_key = fct_sighting.location_key
    JOIN dim_dataset ON dim_dataset.dataset_key = fct_sighting.dataset_key
    WHERE dim_location.in_bay
      AND dim_dataset.kind = 'public'
),

comparison_days AS (
    SELECT
        dim_date.date_key,
        dim_date.holiday_comparison,
        dim_date.compared_holiday,
        dim_date.compared_holiday_date,
        COUNT(bay_public_sightings.record_id) AS public_records
    FROM dim_date
    LEFT JOIN bay_public_sightings ON bay_public_sightings.date_key = dim_date.date_key
    WHERE dim_date.holiday_comparison IS NOT NULL
    GROUP BY
        dim_date.date_key,
        dim_date.holiday_comparison,
        dim_date.compared_holiday,
        dim_date.compared_holiday_date
)

SELECT
    compared_holiday_date AS holiday_date,
    compared_holiday AS holiday,
    SUM(IFF(holiday_comparison = 'Holiday', public_records, NULL)) AS holiday_records,
    AVG(IFF(holiday_comparison = 'Same Weekday', public_records, NULL))
        AS same_weekday_records,
    AVG(IFF(holiday_comparison = 'Weekend', public_records, NULL)) AS weekend_records
FROM comparison_days
GROUP BY compared_holiday_date, compared_holiday
HAVING holiday_records IS NOT NULL
   AND same_weekday_records IS NOT NULL

## The Holiday Ratio
Holiday records ÷ same-weekday records, summed before dividing, so a busy Memorial Day counts for more
than a quiet Columbus Day.

In [ ]:
%%sql
SELECT
    COUNT(*) AS holidays,
    ROUND(AVG(same_weekday_records), 1) AS same_weekday_records_per_day,
    ROUND(AVG(holiday_records), 1) AS holiday_records_per_day,
    ROUND(AVG(weekend_records), 1) AS weekend_records_per_day,
    ROUND(SUM(holiday_records) / SUM(same_weekday_records), 2) AS holiday_ratio,
    ROUND(SUM(weekend_records) / SUM(same_weekday_records), 2) AS weekend_ratio
FROM holiday_compare

## Each Holiday on Its Own

In [ ]:
%%sql
SELECT
    holiday,
    COUNT(*) AS years,
    ROUND(SUM(holiday_records) / SUM(same_weekday_records), 2) AS holiday_ratio
FROM holiday_compare
GROUP BY holiday
ORDER BY holiday_ratio

## How Sure? Resample Holidays
The same holidays drawn at random with replacement 2,000 times, each with its own comparison days,
and the ratio recomputed each time.

In [ ]:
%%sql
WITH numbered_holidays AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY holiday_date) AS holiday_number,
        COUNT(*) OVER () AS total_holidays,
        holiday_records,
        same_weekday_records
    FROM holiday_compare
),

replicates AS (
    SELECT SEQ4() AS replicate
    FROM TABLE(GENERATOR(ROWCOUNT => 2000))
),

-- Each replicate draws as many holidays as there are, at random and with replacement.
resample_draws AS (
    SELECT
        replicates.replicate,
        1 + MOD(
            ABS(HASH(replicates.replicate, numbered_holidays.holiday_number, 20261016)),
            numbered_holidays.total_holidays
        ) AS holiday_number
    FROM replicates
    CROSS JOIN numbered_holidays
),

resampled_ratios AS (
    SELECT
        resample_draws.replicate,
        SUM(numbered_holidays.holiday_records)
            / SUM(numbered_holidays.same_weekday_records) AS holiday_ratio
    FROM resample_draws
    JOIN numbered_holidays
      ON numbered_holidays.holiday_number = resample_draws.holiday_number
    GROUP BY resample_draws.replicate
)

SELECT
    ROUND(PERCENTILE_CONT(0.025) WITHIN GROUP (ORDER BY holiday_ratio), 2) AS range_low,
    ROUND(PERCENTILE_CONT(0.975) WITHIN GROUP (ORDER BY holiday_ratio), 2) AS range_high
FROM resampled_ratios

## Result
A weekday holiday carries **1.30×** the public sightings of the same weekday one week before and after
(95% range 1.11 to 1.54, over the 171 holidays since 2010 that the records reach). That is 9.8 a day on the
holiday, 7.5 on those weekdays and 10.7 on the weekend of the holiday's week, so a holiday goes
(9.8 - 7.5) / (10.7 - 7.5), about seven tenths, of the way from a weekday to a weekend. Christmas Day
gets only 0.27× the public sightings of its comparison weekdays, and Martin Luther King Jr. Day the most,
4.05× theirs.

## Limits
* The 95% range runs from a quarter of the way to a weekend to past a full one.